# Notebook 1 - Method Overview: White-Box Lie Detection with Representation Engineering

## Purpose of this notebook

This notebook explains the full method before running any heavy experiment. The goal is to make the project readable as a course submission: every later notebook follows the same logic introduced here.

The project is inspired by [`mishajw/repeng`](https://github.com/mishajw/repeng). The original repository builds datasets of hidden activations, trains linear probes, and studies whether truth probes generalize across datasets. This repository implements a compact version of that workflow and adds a Colab notebook for comparing several LLMs.

## Research question

We want to test the following hypothesis:

> A language model internally represents whether a candidate answer is true or false, and this truthfulness signal can be read with a simple linear probe.

This is a white-box method because we inspect hidden states directly. We do not ask the model to explain itself, and we do not use generated text as the detector output.


## Step 1 - Build contrast groups

Each dataset is organized into groups. A group corresponds to one question and several candidate answers. Exactly one answer is true, and the other answers are false distractors.

Example:

```text
Question: Which country contains the city Paris?
Candidate answer A: France  -> true
Candidate answer B: Italy   -> false
Candidate answer C: Germany -> false
Candidate answer D: Spain   -> false
```

This grouped design matters. The probe should not merely classify isolated rows. It should rank the true answer above the false answers for the same question.


## Step 2 - Convert each candidate answer into a prompt

We use one fixed prompt template across all models and datasets. This keeps the experiment controlled: the only thing that changes inside a group is the candidate answer.

The model receives a prompt such as:

```text
Consider the correctness of the answer to the following question:

Question: Which country contains the city Paris?
Answer: France
The probability of the answer being correct is
```

The prompt stops before the model generates an answer. We only need the hidden states created while the model reads the prompt.


## Step 3 - Extract hidden states

For each prompt, the model performs one forward pass with `output_hidden_states=True`.

We then take the hidden-state vector at the last token of the prompt for every transformer layer. The last token is useful because it has attended to the full question, the candidate answer, and the instruction text.

If the model has `L` transformer layers and hidden dimension `D`, each prompt becomes an activation tensor of shape:

```text
(L, D)
```

A probe then receives one layer at a time. For example, Phi-2 has 32 layers, so a layer sweep trains and evaluates probes on layer 0, layer 1, ..., layer 31.


## Step 4 - Train a linear truth probe

A probe maps one hidden vector to one scalar score. Higher scores should mean "more likely to be true".

This repository implements four probe methods:

| Method | Full name | Main idea |
|---|---|---|
| `dim` | Difference in Means | Use the normalized difference between the mean true activation and the mean false activation. |
| `lat` | Linear Algebraic Treatment | Use PCA on random pairwise activation differences, following the RepE idea. |
| `lr` | Logistic Regression | Train a supervised linear classifier with scikit-learn. |
| `pca-g` | Grouped PCA | Center activations inside each question group, then use PCA to isolate the within-group truth direction. |

All four methods are linear. This is important: if a simple linear method works, it suggests that truthfulness is represented in a linearly accessible direction of activation space.


## Step 5 - Evaluate with grouped accuracy

The main metric is grouped accuracy.

For each question group:

1. score every candidate answer with the probe,
2. select the candidate with the highest score,
3. count the group as correct if the selected candidate is the true answer.

This is stricter and more meaningful than row-wise binary accuracy. It asks whether the probe can identify the best answer among plausible alternatives.


In [1]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "lie_detector_llm").exists():
            return candidate
    raise RuntimeError("Could not find the project root. Run this notebook from the repository.")

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/imadsharof/Library/Mobile Documents/com~apple~CloudDocs/Cours/MA1/Computing Project/Lie-Detector-for-LLM


## Step 6 - Inspect the available datasets

The next cell builds the lightweight local dataset collection. By default this preview excludes Hugging Face datasets so the overview remains fast. Later notebooks can enable `INCLUDE_HF_DATASETS=True` to add TruthfulQA, ARC, and BoolQ.


In [2]:
import pandas as pd
from lie_detector_llm.datasets import REPE_TEMPLATE, build_dataset_collection

INCLUDE_HF_DATASETS = False

collection = build_dataset_collection(include_hf_datasets=INCLUDE_HF_DATASETS)
df = collection.frame

stats = (
    df.groupby("dataset_name")
    .agg(rows=("dataset_name", "size"), groups=("group_id", "nunique"))
    .reset_index()
    .sort_values("dataset_name")
)

print("Prompt template:")
print()
print(REPE_TEMPLATE)
print()
print("Dataset summary:")
display(stats)
print(f"Total prompts: {len(df)}")
print(f"Total groups : {df['group_id'].nunique()}")

Prompt template:

Consider the correctness of the answer to the following question:

Question: {question}
Answer: {answer}
The probability of the answer being correct is

Dataset summary:


,dataset_name,rows,groups
0,cities,120,30
1,larger_than,60,30
2,qa,120,30
3,repeng_truthful,108,54


Total prompts: 408
Total groups : 144


## Step 7 - Inspect one group

The next cell prints all candidate answers for one question group. This is the exact structure used during evaluation: the probe must rank the true candidate above all false candidates from the same group.


In [3]:
example_group = df["group_id"].iloc[0]
example = df[df["group_id"] == example_group][
    ["dataset_name", "group_id", "question", "answer", "label"]
].reset_index(drop=True)

display(example)
print()
print("Full prompt for the first candidate:")
print()
print(df[df["group_id"] == example_group].iloc[0]["prompt"])

,dataset_name,group_id,question,answer,label
0,qa,qa::12,Who painted the Mona Lisa?,Picasso,False
1,qa,qa::12,Who painted the Mona Lisa?,Leonardo da Vinci,True
2,qa,qa::12,Who painted the Mona Lisa?,Raphael,False
3,qa,qa::12,Who painted the Mona Lisa?,Michelangelo,False



Full prompt for the first candidate:

Consider the correctness of the answer to the following question:

Question: Who painted the Mona Lisa?
Answer: Picasso
The probability of the answer being correct is


## What this notebook establishes

The rest of the project is built from the same pipeline:

1. grouped true/false datasets,
2. fixed prompt template,
3. hidden-state extraction from a selected LLM,
4. linear probe training,
5. grouped accuracy evaluation,
6. transfer, layer, and model comparisons.

The current local baseline model is `microsoft/phi-2`. The Colab notebook later in the sequence is designed for larger models such as Llama 3 8B and optional Llama 3 70B variants.
